[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zshakespear/bd-spectra-onramp/blob/main/notebooks/01_blackbodies.ipynb)

# 1 · Blackbodies and brightness temperature

*Brown Dwarf Spectra On-Ramp · about 35 minutes*

**By the end of this notebook you will be able to:**
- plot Planck curves for 250–2500 K and use Wien's law to find their peaks
- explain why flux on the short-wavelength side of the peak is so sensitive to temperature
- convert an observed spectrum to surface flux using a distance and an assumed radius
- turn a spectrum into a brightness-temperature spectrum and say what it means

> **First time here?** Do *File → Save a copy in Drive* before you change anything (notebook 0 explains why).
> Then run the **Setup** cell below. Run cells in order, top to bottom, with **Shift + Enter**.

In [ ]:
# ▶ SETUP - run this cell first. It is safe to run more than once.
REPO_URL = "https://github.com/zshakespear/bd-spectra-onramp.git"
import os, subprocess, sys
from pathlib import Path

if "google.colab" in sys.modules:                       # running in Google Colab
    ROOT = Path("/content/bd-spectra-onramp")
    if not (ROOT / "bdtools").exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(ROOT)], check=True)
    else:
        subprocess.run(["git", "-C", str(ROOT), "pull", "--ff-only", "--quiet"], check=False)
else:                                                   # running in Jupyter on your own computer
    ROOT = Path.cwd()
    while not (ROOT / "bdtools").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt
import bdtools as bd
bd.setup_notebook()

## 1. Planck curves

A **blackbody** absorbs all light that hits it and glows with a spectrum set *only* by its
temperature — the Planck function B<sub>λ</sub>(T). Brown dwarfs are not blackbodies (that is the
whole point of the later notebooks), but blackbodies are the yardstick we measure them against.
Brown dwarfs span roughly 2500 K (early L) down to 250 K (the coldest Y dwarfs)
(Rustamkulov et al. 2026, Table 1).

In [ ]:
# ✏️ SETTINGS YOU CAN CHANGE
TEMPERATURES = [2500, 1800, 1300, 900, 600, 400, 250]    # kelvin

In [ ]:
wave = np.linspace(0.5, 15, 2000)          # microns
fig, ax = plt.subplots()
colors = plt.cm.inferno(np.linspace(0.1, 0.85, len(TEMPERATURES)))
for T, c in zip(TEMPERATURES, colors):
    ax.plot(wave, bd.blackbody_surface_flux(wave, T), color=c, label=f"{T} K")
    ax.axvline(bd.wien_peak_um(T), color=c, ls=":", lw=1)
ax.axvspan(0.75, 5.0, color="0.85", zorder=0, label="SPHEREx range")
ax.set_yscale("log"); ax.set_ylim(1e-2, 1e7)
ax.set_xlabel("Wavelength [µm]"); ax.set_ylabel(r"Surface flux $\pi B_\lambda$ [W m$^{-2}$ µm$^{-1}$]")
ax.legend(); ax.set_title("Blackbodies (dotted lines: Wien peaks)");

**Wien's law**: the peak of B<sub>λ</sub> is at λ<sub>max</sub> = 2898 µm·K / T.

In [ ]:
for T in TEMPERATURES:
    peak = bd.wien_peak_um(T)
    where = "inside" if 0.75 <= peak <= 5.0 else ("longward of" if peak > 5 else "shortward of")
    print(f"T = {T:5d} K   peak at {peak:5.2f} um   ({where} the SPHEREx range)")

Notice: for L dwarfs (≈1300–2500 K) the peak sits *inside* the SPHEREx range. For Y dwarfs and the
very coldest T dwarfs (below about 580 K) the peak is beyond 5 µm, so **everything SPHEREx sees is on
the short-wavelength side of the peak** — the steep side. That matters a lot, as the next section shows.

## 2. Why a small temperature change makes a huge flux change

**Predict, then run.** Take two blackbodies, 1000 K and 1100 K (10% hotter). At 1.25 µm, how
many times brighter is the hotter one? At 4.5 µm? Write your guesses down first.

In [ ]:
for lam in [1.25, 2.2, 4.5]:
    ratio = bd.planck_lambda(lam, 1100) / bd.planck_lambda(lam, 1000)
    print(f"at {lam:4.2f} um: 1100 K is {ratio:4.2f} times brighter than 1000 K")

On the short-wavelength side of the peak, B<sub>λ</sub> falls off *exponentially*
(∝ e<sup>−hc/λkT</sup>), so a modest temperature contrast becomes an **enormous flux contrast**.
At long wavelengths the dependence is gentler. This is why:

* we plot spectra on **log** axes, and
* we often convert flux into **brightness temperature**, which turns those huge flux ratios back
  into modest, comparable temperature differences.

## 3. From observed flux to surface flux

The flux we measure is diluted by distance. Light leaving a sphere of radius R spreads over a
sphere of radius d by the time it reaches us, so

  F<sub>observed</sub> = F<sub>surface</sub> × (R / d)².

Distances come from parallaxes (stored with each reference spectrum). Radii are **not measured**;
brown dwarfs are all about the size of Jupiter (≈0.8–1.3 R<sub>Jup</sub>), so we *assume* one.

In [ ]:
# ✏️ SETTINGS YOU CAN CHANGE
SPECTRAL_TYPE = "L5"
RADIUS_RJUP = 1.0          # assumed radius, in Jupiter radii

In [ ]:
spec = bd.load_reference(SPECTRAL_TYPE)
d = spec.meta["distance_pc"]
surface = bd.observed_to_surface(spec.flux, distance_pc=d, radius_rjup=RADIUS_RJUP)
print(f"{spec.name}: distance {d} pc; surface flux is {surface[0] / spec.flux[0]:.3g} times the observed flux")

## 4. Brightness temperature

The **brightness temperature** T<sub>b</sub>(λ) is the temperature a blackbody would need to emit the
same surface flux *at that wavelength*. For a real blackbody T<sub>b</sub> is the same at every λ.
For a brown dwarf it is not — and the way it varies tells us about the atmosphere.

In [ ]:
tb = bd.brightness_temperature(spec.wave_um, surface)
fig, axes = plt.subplots(2, 1, sharex=True, figsize=(9, 7))
axes[0].plot(spec.wave_um, spec.flux, "o-", ms=3, color="k"); axes[0].set_yscale("log")
axes[0].set_ylabel(r"observed $F_\lambda$ [W m$^{-2}$ µm$^{-1}$]")
axes[1].plot(spec.wave_um, tb, "o-", ms=3, color=bd.LINE_COLORS[1])
axes[1].set_ylabel("brightness temperature [K]"); axes[1].set_xlabel("Wavelength [µm]")
axes[0].set_title(f"{spec.name}, assuming R = {RADIUS_RJUP} R_Jup");

Read the bottom panel like a map of *depth*. Where T<sub>b</sub> is **high**, we see light from deep, hot
layers (these are the "windows"). Where T<sub>b</sub> is **low**, something absorbs the deep light and we
only see cooler, higher layers. Notebook 2 explains why.

**A caution that worked well in past versions of this course:** comparing fluxes at different
wavelengths and saying "brighter here, so hotter here" can mislead, because the Planck function
itself depends on wavelength. The flux peak of a blackbody is not where it is "hottest". Brightness
temperature fixes this: it is the one quantity you can compare fairly across wavelengths.

### Exercise 1.1
At about what wavelength is T<sub>b</sub> highest for this spectrum, and roughly what is it? Where is it lowest?

<details><summary><b>Answer</b> (try first, then click)</summary>

For the L5 the highest T<sub>b</sub> is in the near-infrared windows (around the J peak, 1.2–1.3 µm)
and the lowest in the absorption troughs (e.g. near 1.4 µm, 1.9 µm, 2.7 µm) and at long
wavelengths. Exact numbers depend on the assumed radius — see the next exercise.

</details>

### Exercise 1.2
Set `RADIUS_RJUP = 0.8`, then 1.2, and re-run. Does the *shape* of the T<sub>b</sub> curve change, or only its level?
Which wavelengths change most in kelvin?

<details><summary><b>Answer</b> (try first, then click)</summary>

Only the level shifts — the pattern of high and low T<sub>b</sub> stays the same. Because surface flux
∝ 1/R², a smaller radius means more surface flux and a higher T<sub>b</sub>. The shift in kelvin is larger at
long wavelengths, where B<sub>λ</sub> depends less steeply on T, so a given flux change needs a bigger
temperature change. This is why we treat absolute T<sub>b</sub> values as rough but trust the pattern.

</details>

### Exercise 1.3
Use `bd.brightness_temperature` to find the T<sub>b</sub> a blackbody at 1500 K has at 1 µm and at 4 µm.
(Hint: first compute its surface flux with `bd.blackbody_surface_flux`.)

<details><summary><b>Answer</b> (try first, then click)</summary>

```python
for lam in [1.0, 4.0]:
    print(lam, bd.brightness_temperature(lam, bd.blackbody_surface_flux(lam, 1500)))
```
Both are 1500 K — for a true blackbody T<sub>b</sub> is the same at every wavelength. Any variation of
T<sub>b</sub> with wavelength in a real spectrum is information about the atmosphere.

</details>

## Summary

- Wien's law: λ<sub>max</sub> = 2898 µm K / T. L dwarfs peak inside 0.75–5 µm; Y dwarfs peak beyond it.
- Short of the peak, flux depends exponentially on temperature: small ΔT, huge flux ratio.
- Surface flux = observed flux × (d/R)²; R is assumed (~1 R<sub>Jup</sub>).
- Brightness temperature lets you compare wavelengths fairly; high T<sub>b</sub> = seeing deep, low T<sub>b</sub> = seeing high.

**Field guide:** the pages linked above are the short versions to keep open while you work.

## Self-check

Answer these without scrolling up. If one is hard, revisit that section.

1. Why is SPHEREx's whole range on the steep side of the Planck curve for a Y dwarf?
<details><summary><b>Answer</b> (try first, then click)</summary>

Its temperature is below ~580 K, so the Wien peak is beyond 5 µm.

</details>

2. A 10% hotter blackbody is about how many times brighter at 1.25 µm? At 4.5 µm?
<details><summary><b>Answer</b> (try first, then click)</summary>

About 2.8× at 1.25 µm and about 1.3× at 4.5 µm.

</details>

3. What two numbers do you need to turn observed flux into surface flux, and which is assumed?
<details><summary><b>Answer</b> (try first, then click)</summary>

Distance (measured from parallax) and radius (assumed, ~1 R<sub>Jup</sub>).

</details>

4. What does a low brightness temperature at some wavelength tell you?
<details><summary><b>Answer</b> (try first, then click)</summary>

Light at that wavelength comes from higher, cooler layers — deep light is being absorbed.

</details>

**Next:** [Notebook 2 · Opacity and the photosphere](02_opacity_photosphere.ipynb)